In [18]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

In [19]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [20]:
BASE_DIR = "output_dataset"

image_dir = os.path.join(BASE_DIR, "images")
crop_dir = os.path.join(BASE_DIR, "crop_mask")
damage_dir = os.path.join(BASE_DIR, "damage_mask")

valid_ext = (".png", ".jpg", ".jpeg")

files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(valid_ext)])

print("Total images:", len(files))
print("Sample file:", files[0])

Total images: 541
Sample file: IMG_2490_jpeg.rf.f884e7cf1de233949f08d4d62668bbb1.jpg


In [21]:
from sklearn.model_selection import train_test_split

train_files, temp_files = train_test_split(files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

print("Train:", len(train_files))
print("Validation:", len(val_files))
print("Test:", len(test_files))

Train: 378
Validation: 81
Test: 82


In [22]:
class CropDataset(Dataset):
    def __init__(self, files, img_dir, crop_dir, damage_dir, transform=None):
        self.files = files
        self.img_dir = img_dir
        self.crop_dir = crop_dir
        self.damage_dir = damage_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]

        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f)), cv2.COLOR_BGR2RGB)
        crop = cv2.imread(os.path.join(self.crop_dir, f), 0)
        damage = cv2.imread(os.path.join(self.damage_dir, f), 0)

        # Binary masks
        crop = (crop > 127).astype(np.float32)
        damage = (damage > 127).astype(np.float32)

        # Restrict damage to crop only
        damage = damage * crop

        if self.transform:
            aug = self.transform(image=img, masks=[crop, damage])
            img = aug["image"]
            crop, damage = aug["masks"]

        crop = crop.unsqueeze(0)
        damage = damage.unsqueeze(0)

        return img, crop, damage

In [23]:
train_transform = A.Compose([
    A.Resize(256,256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(256,256),
    A.Normalize(),
    ToTensorV2()
])

In [24]:
train_dataset = CropDataset(
    train_files, image_dir, crop_dir, damage_dir, transform=train_transform
)

val_dataset = CropDataset(
    val_files, image_dir, crop_dir, damage_dir, transform=val_transform
)

test_dataset = CropDataset(
    test_files, image_dir, crop_dir, damage_dir, transform=val_transform
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 378
Val dataset: 81
Test dataset: 82


In [25]:
batch_size = 2

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print("DataLoaders ready")

DataLoaders ready


In [26]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
)

model = model.to(device)

print("Attention UNet ready")

Attention UNet ready


In [27]:
bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

print("Loss function ready")

Loss function ready


In [28]:
def iou_score(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    intersection = (pred * target).sum()
    union = (pred + target).sum() - intersection

    return (intersection + 1e-6) / (union + 1e-6)


def dice_score(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    intersection = (pred * target).sum()

    return (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)


print("Metrics ready")

Metrics ready


In [30]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',   # maximize IoU
    patience=3,
    factor=0.5
)

print("Optimizer and scheduler ready")

Optimizer and scheduler ready


In [31]:
num_epochs = 40
best_val_iou = 0
patience = 8
early_stop_counter = 0

train_losses = []
val_ious = []
val_dices = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------- TRAIN --------
    model.train()
    total_loss = 0

    for images, _, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # -------- VALIDATION --------
    model.eval()
    total_iou = 0
    total_dice = 0

    with torch.no_grad():
        for images, _, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            total_iou += iou_score(outputs, masks).item()
            total_dice += dice_score(outputs, masks).item()

    avg_val_iou = total_iou / len(val_loader)
    avg_val_dice = total_dice / len(val_loader)

    val_ious.append(avg_val_iou)
    val_dices.append(avg_val_dice)

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val IoU: {avg_val_iou:.4f} | Dice: {avg_val_dice:.4f}")

    # Scheduler step
    scheduler.step(avg_val_iou)

    # Save best model
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        torch.save(model.state_dict(), "best_attunet_exp1.pth")
        print("✅ Best model saved!")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    # Early stopping
    if early_stop_counter >= patience:
        print("⛔ Early stopping triggered")
        break


Epoch 1/40
Train Loss: 1.2105
Val IoU: 0.3481 | Dice: 0.5033
✅ Best model saved!

Epoch 2/40
Train Loss: 0.8374
Val IoU: 0.4468 | Dice: 0.6081
✅ Best model saved!

Epoch 3/40
Train Loss: 0.7433
Val IoU: 0.4342 | Dice: 0.5939

Epoch 4/40
Train Loss: 0.6989
Val IoU: 0.4290 | Dice: 0.5905

Epoch 5/40
Train Loss: 0.6649
Val IoU: 0.4960 | Dice: 0.6581
✅ Best model saved!

Epoch 6/40
Train Loss: 0.6401
Val IoU: 0.4656 | Dice: 0.6242

Epoch 7/40
Train Loss: 0.6313
Val IoU: 0.4920 | Dice: 0.6518

Epoch 8/40
Train Loss: 0.6076
Val IoU: 0.5063 | Dice: 0.6656
✅ Best model saved!

Epoch 9/40
Train Loss: 0.6004
Val IoU: 0.4823 | Dice: 0.6425

Epoch 10/40
Train Loss: 0.5813
Val IoU: 0.5134 | Dice: 0.6743
✅ Best model saved!

Epoch 11/40
Train Loss: 0.5752
Val IoU: 0.5162 | Dice: 0.6755
✅ Best model saved!

Epoch 12/40
Train Loss: 0.5532
Val IoU: 0.5274 | Dice: 0.6856
✅ Best model saved!

Epoch 13/40
Train Loss: 0.5415
Val IoU: 0.5206 | Dice: 0.6787

Epoch 14/40
Train Loss: 0.5309
Val IoU: 0.5219 | 

In [32]:
model.load_state_dict(torch.load("best_attunet_exp1.pth"))
model.eval()

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [33]:
thresholds = [0.3, 0.4, 0.5, 0.6]

for t in thresholds:
    total_iou = 0
    
    with torch.no_grad():
        for images, _, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            total_iou += iou_score(outputs, masks, threshold=t).item()

    avg_iou = total_iou / len(val_loader)
    print(f"Threshold {t}: IoU = {avg_iou:.4f}")

Threshold 0.3: IoU = 0.5450
Threshold 0.4: IoU = 0.5455
Threshold 0.5: IoU = 0.5431
Threshold 0.6: IoU = 0.5387


In [40]:
train_transform = A.Compose([
    A.Resize(320, 320),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(320, 320),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [41]:
train_dataset = CropDataset(train_files, image_dir, crop_dir, damage_dir, transform=train_transform)
val_dataset = CropDataset(val_files, image_dir, crop_dir, damage_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

In [42]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
).to(device)

In [43]:
num_epochs = 25
best_val_iou = 0
patience = 8
early_stop_counter = 0

train_losses = []
val_ious = []
val_dices = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------- TRAIN --------
    model.train()
    total_loss = 0

    for images, _, masks in train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # -------- VALIDATION --------
    model.eval()
    total_iou = 0
    total_dice = 0

    with torch.no_grad():
        for images, _, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            total_iou += iou_score(outputs, masks).item()
            total_dice += dice_score(outputs, masks).item()

    avg_val_iou = total_iou / len(val_loader)
    avg_val_dice = total_dice / len(val_loader)

    val_ious.append(avg_val_iou)
    val_dices.append(avg_val_dice)

    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val IoU: {avg_val_iou:.4f} | Dice: {avg_val_dice:.4f}")

    # Scheduler step
    scheduler.step(avg_val_iou)

    # Save best model
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        torch.save(model.state_dict(), "best_attunet_exp1.pth")
        print("✅ Best model saved!")
        early_stop_counter = 0
    else:
        early_stop_counter += 1

    # Early stopping
    if early_stop_counter >= patience:
        print("⛔ Early stopping triggered")
        break


Epoch 1/25
Train Loss: 1.8495
Val IoU: 0.1044 | Dice: 0.1857
✅ Best model saved!

Epoch 2/25
Train Loss: 1.8464
Val IoU: 0.1045 | Dice: 0.1858
✅ Best model saved!

Epoch 3/25


KeyboardInterrupt: 

# AGarin

In [44]:
!pip install segmentation-models-pytorch albumentations opencv-python

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

In [45]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: mps


In [46]:
BASE_DIR = "output_dataset"

image_dir = os.path.join(BASE_DIR, "images")
crop_dir = os.path.join(BASE_DIR, "crop_mask")
damage_dir = os.path.join(BASE_DIR, "damage_mask")

files = sorted([f for f in os.listdir(image_dir) if f.endswith((".jpg",".png",".jpeg"))])

print("Total images:", len(files))

Total images: 541


In [47]:
from sklearn.model_selection import train_test_split

train_files, temp_files = train_test_split(files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

print(len(train_files), len(val_files), len(test_files))

378 81 82


In [48]:
class CropDataset(Dataset):
    def __init__(self, files, img_dir, crop_dir, damage_dir, transform=None):
        self.files = files
        self.img_dir = img_dir
        self.crop_dir = crop_dir
        self.damage_dir = damage_dir
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]

        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f)), cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0   # 🔥 FIX

        crop = cv2.imread(os.path.join(self.crop_dir, f), 0)
        damage = cv2.imread(os.path.join(self.damage_dir, f), 0)

        crop = (crop > 127).astype(np.float32)
        damage = (damage > 127).astype(np.float32)

        damage = damage * crop

        if self.transform:
            aug = self.transform(image=img, masks=[crop, damage])
            img = aug["image"]
            crop, damage = aug["masks"]

        crop = crop.unsqueeze(0)
        damage = damage.unsqueeze(0)

        return img, crop, damage

In [49]:
train_transform = A.Compose([
    A.Resize(320,320),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0,0,0), std=(1,1,1)),   # 🔥 safe normalize
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(320,320),
    A.Normalize(mean=(0,0,0), std=(1,1,1)),
    ToTensorV2()
])

In [50]:
train_dataset = CropDataset(train_files, image_dir, crop_dir, damage_dir, transform=train_transform)
val_dataset   = CropDataset(val_files, image_dir, crop_dir, damage_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

print("Data ready")

Data ready


In [51]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
).to(device)

print("Model ready")

Model ready


In [52]:
bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum()
    return 1 - (2*inter + 1e-6)/(pred.sum()+target.sum()+1e-6)

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

In [53]:
def iou_score(pred, target, threshold=0.5):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    inter = (pred * target).sum()
    union = (pred + target).sum() - inter

    return (inter + 1e-6)/(union + 1e-6)

In [54]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [55]:
images, _, masks = next(iter(train_loader))
images = images.to(device)

outputs = model(images)

print("Output range:", outputs.min().item(), outputs.max().item())

Output range: -9.868962287902832 9.3016357421875


In [56]:
num_epochs = 25
best_iou = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    model.train()
    train_loss = 0

    for img, _, mask in train_loader:
        img, mask = img.to(device), mask.to(device)

        optimizer.zero_grad()
        out = model(img)
        loss = loss_fn(out, mask)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() / len(train_loader)

    # validation
    model.eval()
    val_iou = 0

    with torch.no_grad():
        for img, _, mask in val_loader:
            img, mask = img.to(device), mask.to(device)
            out = model(img)
            val_iou += iou_score(out, mask).item()

    val_iou /= len(val_loader)

    print(f"Loss: {train_loss:.4f} | Val IoU: {val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved best model")


Epoch 1/25
Loss: 193.073 | Val IoU: 0.4341
✅ Saved best model

Epoch 2/25
Loss: 142.461 | Val IoU: 0.4558
✅ Saved best model

Epoch 3/25
Loss: 132.486 | Val IoU: 0.4661
✅ Saved best model

Epoch 4/25
Loss: 126.400 | Val IoU: 0.4937
✅ Saved best model

Epoch 5/25
Loss: 121.427 | Val IoU: 0.4864

Epoch 6/25
Loss: 119.697 | Val IoU: 0.5148
✅ Saved best model

Epoch 7/25
Loss: 116.003 | Val IoU: 0.5056

Epoch 8/25
Loss: 113.152 | Val IoU: 0.4870

Epoch 9/25
Loss: 109.809 | Val IoU: 0.4780

Epoch 10/25
Loss: 107.638 | Val IoU: 0.4547

Epoch 11/25
Loss: 107.688 | Val IoU: 0.5089

Epoch 12/25
Loss: 104.451 | Val IoU: 0.5236
✅ Saved best model

Epoch 13/25
Loss: 101.279 | Val IoU: 0.5013

Epoch 14/25
Loss: 100.860 | Val IoU: 0.5225

Epoch 15/25
Loss: 98.124 | Val IoU: 0.4914

Epoch 16/25
Loss: 98.177 | Val IoU: 0.4918

Epoch 17/25
Loss: 95.201 | Val IoU: 0.4644

Epoch 18/25
Loss: 94.326 | Val IoU: 0.5277
✅ Saved best model

Epoch 19/25
Loss: 91.812 | Val IoU: 0.5345
✅ Saved best model

Epoch 